# village-seg-baseline, round 2 — three villages plus annotated drone frames

Continues from the round-1 checkpoint (`model-imagenet.pt`, the better of the two
round-1 runs). New since round 1:

- `ua-village-annotated` now has **three** villages: birky (4 dates), bushcha and
  dubrova (2 dates each). The new two are woody-dominated (50-54% woody), birky is
  buildings and tracks.
- `ua-drone-annotated`: **44 hand-labelled 2025 drone frames** (background, track,
  woody; no buildings), 0.014-0.038 m/px.

**Hold-outs** — three, each answering a different question:

| set | what | answers |
|---|---|---|
| `birky-strip` | top quarter of birky, all 4 dates (as round 1) | comparable to round 1 |
| `bushcha` | the whole village, both dates, never trained on | **does it generalise to a new village?** |
| `drone-val` | drone time blocks 0 and 4 (16 frames) | **does it now work on drone frames?** |

Drone frames 1 s apart overlap almost completely, so the drone split is by
**ground footprint**, not by frame: any training frame whose footprint touches a
held-out footprint is dropped (block 1, 6 frames). Train = blocks 2 + 3 (22
frames, including a 22 -> 54 m climb).

**Step 0 is the round-1 model** — evaluating it on the new hold-outs is the
"before" of this round.

**Domains are mixed by step**, not within a batch: each step is either a
satellite batch (4 x 512 px at 0.385 m/px) or a drone batch (8 x 256 px at
~`drone_gsd` m/px) — same pixel count per step. Drone frames are presented at
`drone_gsd` with +-25% scale jitter, not at the satellite GSD: at 0.385 m/px a
frame is 130 px and a dirt track is 5 px wide (see `drone.ipynb`).

`PYTORCH_ENABLE_MPS_FALLBACK=1 papermill pocs/village-seg-baseline/round2.ipynb pocs/village-seg-baseline/runs/round2.ipynb`

In [ ]:
# papermill parameters
SLUG = "village-seg-baseline"
init_ckpt = "model-imagenet.pt"
tag = "round2"
sat_holdout_village = "bushcha-50p2422-25p8305"
birky = "birky-49p9105-23p9192"
drone_val_blocks = [0, 4]
drone_p = 0.5          # share of steps that are drone batches, 0 = satellite only
drone_gsd = 0.10       # m/px the drone frames are presented at
sat_crop, sat_batch = 512, 4
drone_crop, drone_batch = 256, 8
steps = 1500
eval_every = 250
lr = 2e-4              # lower than round 1: fine-tuning a trained model
rare_crop_p = 0.3
gap = 64
seed = 0

## Setup

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import json, math, time, warnings
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from shapely import affinity
from shapely.geometry import Polygon

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None
assert Path("datasets").is_dir(), "run from src/ — cwd is " + str(Path.cwd())

from tools.annotator.refs._infer import tiled_logits

DATA = Path("data/derived")
VIL = DATA / "ua-village-annotated"
DRN = DATA / "ua-drone-annotated"
HERE = Path("pocs") / SLUG
FIGS = HERE / "figures"

CLASSES = ["background", "building", "track", "woody", "field", "water"]
IGNORE = 255
PAL = np.array([[40, 40, 40], [230, 30, 30], [255, 215, 0], [20, 160, 40],
                [200, 140, 255], [30, 110, 255]], np.uint8)
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)
DEV = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(seed); np.random.seed(seed)
print("torch", torch.__version__, "| device", DEV, "| init", init_ckpt)


def norm(img):
    return (np.asarray(img, np.float32) / 255.0 - MEAN) / STD


def colour(lab):
    out = np.zeros((*lab.shape, 3), np.uint8)
    ok = lab != IGNORE
    out[ok] = PAL[lab[ok]]
    out[~ok] = 255
    return out

## Stage 0 — satellite data

Same loading as round 1: labels are in each image's frame; the strip a date's
alignment uncovered is set to ignore.

In [ ]:
def uncovered(shape, off):
    dx, dy = off
    h, w = shape
    m = np.zeros(shape, bool)
    if dx > 0: m[:, w - dx:] = True
    if dx < 0: m[:, :-dx] = True
    if dy > 0: m[h - dy:, :] = True
    if dy < 0: m[:-dy, :] = True
    return m


vrecs = sorted((json.loads(l) for l in (VIL / "samples.jsonl").read_text().splitlines() if l.strip()),
               key=lambda r: (r["location"], r["date"]))
sat = []
for r in vrecs:
    img = np.array(Image.open(DATA / r["image_file"]).convert("RGB"))
    lab = np.array(Image.open(VIL / r["label_file"]))
    assert img.shape[:2] == lab.shape, (r["location"], img.shape, lab.shape)
    lab[uncovered(lab.shape, r["image_offset_px"])] = IGNORE
    sat.append(dict(loc=r["location"], date=r["date"], img=img, lab=lab))

print(f"{'location':>26s} {'date':>11s} " + " ".join(f"{c[:6]:>7s}" for c in CLASSES))
for s in sat:
    print(f"{s['loc']:>26s} {s['date']:>11s} " + " ".join(f"{(s['lab'] == i).mean():>7.1%}" for i in range(6)))

### Satellite split

- `bushcha`: held out whole.
- `birky`: rows 0:640 held out on every date (round 1's strip), 64 px gap.
- `dubrova` and the rest of birky: training.

In [ ]:
BIRKY_VAL = (0, 640)   # rows; round 1 chose this strip by rule, kept fixed for comparability

for s in sat:
    h, w = s["lab"].shape
    banned = np.zeros((h, w), bool)
    if s["loc"] == sat_holdout_village:
        banned[:] = True
    elif s["loc"] == birky:
        banned[:BIRKY_VAL[1] + gap] = True
    s["banned"] = banned

sat_train = [s for s in sat if not s["banned"].all()]
print("training images:", [(s["loc"][:8], s["date"]) for s in sat_train])
print("bushcha val:", [s["date"] for s in sat if s["loc"] == sat_holdout_village])


def valid_corners(banned, crop):
    ii = np.pad(banned.astype(np.int32).cumsum(0).cumsum(1), ((1, 0), (1, 0)))
    hits = ii[crop:, crop:] - ii[:-crop, crop:] - ii[crop:, :-crop] + ii[:-crop, :-crop]
    return hits == 0


for s in sat_train:
    ok = valid_corners(s["banned"], sat_crop)
    s["tl"] = np.argwhere(ok)
    s["ok"] = ok

# rare-class centres across all training images: (image idx, y, x)
sat_centres = {}
for c in (1, 2, 3, 5):
    pts = []
    for k, s in enumerate(sat_train):
        ys, xs = np.nonzero(s["lab"] == c)
        H, W = s["lab"].shape
        t = np.clip(ys - sat_crop // 2, 0, H - sat_crop); l = np.clip(xs - sat_crop // 2, 0, W - sat_crop)
        keep = s["ok"][t, l]
        sel = np.nonzero(keep)[0][::7]                 # thin: neighbouring pixels give the same crop
        pts.append(np.c_[np.full(len(sel), k), ys[sel], xs[sel]])
    sat_centres[c] = np.concatenate(pts)
    print(f"  {CLASSES[c]:>8s}: {len(sat_centres[c]):>8,} centre candidates")

## Stage 1 — drone data and its footprint split

In [ ]:
drecs = sorted((json.loads(l) for l in (DRN / "samples.jsonl").read_text().splitlines() if l.strip()),
               key=lambda r: r["position"]["timestamp_ms"])
lat0, lon0 = drecs[0]["position"]["lat"], drecs[0]["position"]["lon"]
drone, blk, prev = [], 0, None
for r in drecs:
    p = r["position"]
    if prev is not None and p["timestamp_ms"] - prev > 5000:
        blk += 1
    prev = p["timestamp_ms"]
    x = (p["lon"] - lon0) * 111320 * math.cos(math.radians(lat0)); y = (p["lat"] - lat0) * 110540
    w = 2 * p["alt_m"] * math.tan(math.radians(31.5)); h = 2 * p["alt_m"] * math.tan(math.radians(17.5))
    fp = affinity.translate(affinity.rotate(Polygon([(-w/2, -h/2), (w/2, -h/2), (w/2, h/2), (-w/2, h/2)]),
                                            -p["yaw_deg"], origin=(0, 0)), x, y)
    drone.append(dict(name=r["location"], block=blk, mpp=r["mpp"], alt=p["alt_m"], fp=fp,
                      img=np.array(Image.open(DATA / r["image_file"]).convert("RGB")),
                      lab=np.array(Image.open(DRN / r["label_file"]))))

val_union = None
for d in drone:
    if d["block"] in drone_val_blocks:
        val_union = d["fp"] if val_union is None else val_union.union(d["fp"])
for d in drone:
    d["split"] = ("val" if d["block"] in drone_val_blocks
                  else "dropped" if d["fp"].intersects(val_union) else "train")

print(f"{'blk':>3s} {'n':>3s} {'split':>8s} {'alt m':>11s} {'track':>6s} {'woody':>6s}")
for b in sorted({d["block"] for d in drone}):
    ds = [d for d in drone if d["block"] == b]
    print(f"{b:>3d} {len(ds):>3d} {','.join(sorted({d['split'] for d in ds})):>8s} "
          f"{min(d['alt'] for d in ds):>5.0f}-{max(d['alt'] for d in ds):<5.0f} "
          f"{np.mean([(d['lab'] == 2).mean() for d in ds]):>6.1%} {np.mean([(d['lab'] == 3).mean() for d in ds]):>6.1%}")
drone_train = [d for d in drone if d["split"] == "train"]
drone_val = [d for d in drone if d["split"] == "val"]
print(f"\ndrone train {len(drone_train)}, val {len(drone_val)}, dropped {sum(d['split'] == 'dropped' for d in drone)}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
col = {"train": "tab:blue", "val": "tab:red", "dropped": "0.6"}
for d in drone:
    xs, ys = d["fp"].exterior.xy
    ax.plot(xs, ys, color=col[d["split"]], lw=.8)
for k, c in col.items():
    ax.plot([], [], color=c, label=k)
ax.set_aspect("equal"); ax.legend(); ax.set_xlabel("east, m"); ax.set_ylabel("north, m")
ax.set_title("drone footprints: no training footprint touches a held-out one")
plt.tight_layout(); plt.savefig(FIGS / f"{tag}-drone-split.png", dpi=100); plt.show()

## Stage 2 — samplers

In [ ]:
def augment(img, lab, rng):
    k = rng.integers(4)
    img, lab = np.rot90(img, k), np.rot90(lab, k)
    if rng.random() < 0.5:
        img, lab = img[:, ::-1], lab[:, ::-1]
    x = np.asarray(img, np.float32) / 255.0
    x = x * (1 + rng.uniform(-0.15, 0.15) + rng.uniform(-0.05, 0.05, 3))
    m = x.mean(); x = (x - m) * (1 + rng.uniform(-0.15, 0.15)) + m
    x = (x - MEAN) / STD
    return x.transpose(2, 0, 1).copy(), lab.copy()


def sat_sample(rng):
    if rng.random() < rare_crop_p:
        c = rng.choice([c for c in sat_centres if len(sat_centres[c])])
        k, y, x = sat_centres[c][rng.integers(len(sat_centres[c]))]
        s = sat_train[k]; H, W = s["lab"].shape
        t, l = np.clip(y - sat_crop // 2, 0, H - sat_crop), np.clip(x - sat_crop // 2, 0, W - sat_crop)
    else:
        s = sat_train[rng.integers(len(sat_train))]
        t, l = s["tl"][rng.integers(len(s["tl"]))]
    return augment(s["img"][t:t + sat_crop, l:l + sat_crop], s["lab"][t:t + sat_crop, l:l + sat_crop], rng)


def drone_sample(rng):
    d = drone_train[rng.integers(len(drone_train))]
    g = drone_gsd * rng.uniform(0.8, 1.25)
    n = int(round(drone_crop * g / d["mpp"]))            # native px that become one crop
    H, W = d["lab"].shape
    if rng.random() < rare_crop_p and (d["lab"] == 2).any() | (d["lab"] == 3).any():
        c = rng.choice([c for c in (2, 3) if (d["lab"] == c).any()])
        ys, xs = np.nonzero(d["lab"] == c); j = rng.integers(len(ys))
        cy, cx = ys[j], xs[j]
    else:
        cy, cx = rng.integers(H), rng.integers(W)
    # keep the crop inside the frame where it fits; centre it (padding both sides) where it does not
    t = int(np.clip(cy - n // 2, 0, H - n)) if n <= H else (H - n) // 2
    l = int(np.clip(cx - n // 2, 0, W - n)) if n <= W else (W - n) // 2
    img = np.zeros((n, n, 3), np.uint8); lab = np.full((n, n), IGNORE, np.uint8)
    ys, xs = slice(max(t, 0), min(t + n, H)), slice(max(l, 0), min(l + n, W))
    img[ys.start - t:ys.stop - t, xs.start - l:xs.stop - l] = d["img"][ys, xs]
    lab[ys.start - t:ys.stop - t, xs.start - l:xs.stop - l] = d["lab"][ys, xs]
    img = np.asarray(Image.fromarray(img).resize((drone_crop, drone_crop), Image.BILINEAR))
    lab = np.asarray(Image.fromarray(lab).resize((drone_crop, drone_crop), Image.NEAREST))
    return augment(img, lab, rng)


def batch_of(fn, rng, n):
    xs, ys = zip(*(fn(rng) for _ in range(n)))
    return torch.from_numpy(np.stack(xs)).float(), torch.from_numpy(np.stack(ys)).long()


rng_show = np.random.default_rng(1)
fig, axes = plt.subplots(2, 8, figsize=(20, 5.2))
for i in range(8):
    fn = sat_sample if i < 4 else drone_sample
    x, y = fn(rng_show)
    axes[0, i].imshow(np.clip(x.transpose(1, 2, 0) * STD + MEAN, 0, 1)); axes[1, i].imshow(colour(y))
    axes[0, i].set_title("satellite" if i < 4 else "drone", fontsize=9)
    for a in axes[:, i]: a.axis("off")
fig.suptitle("training crops as the model sees them — sanity check"); plt.tight_layout(); plt.show()

## Stage 3 — evaluation

IoU per class, intersection and union summed over the set. `mIoU` is the mean
over classes that **have labelled pixels in that set** (drone-val has no
building, field or water; no set has field). Cross-date agreement is reported
for the two satellite sets.

In [ ]:
@torch.no_grad()
def predict_big(model, img):
    model.eval()
    return tiled_logits(model, norm(img), 512, 384, len(CLASSES), DEV).argmax(0).astype(np.uint8)


@torch.no_grad()
def predict_small(model, img):
    model.eval()
    x = torch.from_numpy(norm(img).transpose(2, 0, 1))[None]
    h, w = x.shape[2:]
    H, W = -(-h // 32) * 32, -(-w // 32) * 32
    x = F.pad(x, (0, W - w, 0, H - h), mode="reflect")
    return model(x.float().to(DEV))[0, :, :h, :w].argmax(0).cpu().numpy().astype(np.uint8)


def drone_eval_pair(d):
    """Frame and label at drone_gsd, the scale the model is trained to see."""
    s = d["mpp"] / drone_gsd
    size = (round(d["img"].shape[1] * s), round(d["img"].shape[0] * s))
    return (np.asarray(Image.fromarray(d["img"]).resize(size, Image.LANCZOS)),
            np.asarray(Image.fromarray(d["lab"]).resize(size, Image.NEAREST)))


VAL = {
    "birky-strip": [(s["img"][:BIRKY_VAL[1]], s["lab"][:BIRKY_VAL[1]], "big") for s in sat if s["loc"] == birky],
    "bushcha": [(s["img"], s["lab"], "big") for s in sat if s["loc"] == sat_holdout_village],
    "drone-val": [(*drone_eval_pair(d), "small") for d in drone_val],
}


def scores(preds, labs):
    I = np.zeros(6); U = np.zeros(6); npx = np.zeros(6)
    for p, t in zip(preds, labs):
        ok = t != IGNORE
        for c in range(6):
            I[c] += ((p == c) & (t == c) & ok).sum(); U[c] += (((p == c) | (t == c)) & ok).sum()
            npx[c] += ((t == c) & ok).sum()
    iou = {CLASSES[c]: float(I[c] / U[c]) for c in range(6) if npx[c] > 0}
    return {"iou": iou, "miou": float(np.mean(list(iou.values())))}


def agreement(arrs):
    pairs = [(i, j) for i in range(len(arrs)) for j in range(i + 1, len(arrs))]
    return float(np.mean([(arrs[i] == arrs[j]).mean() for i, j in pairs])) if pairs else None


def evaluate(model):
    out, preds = {}, {}
    for name, items in VAL.items():
        ps = [predict_big(model, im) if kind == "big" else predict_small(model, im) for im, _, kind in items]
        out[name] = scores(ps, [lb for _, lb, _ in items])
        if name != "drone-val":
            out[name]["agreement"] = agreement(ps)
        preds[name] = ps
    return out, preds


for name, items in VAL.items():
    labs = [lb for _, lb, _ in items]
    print(f"{name:>12s}: {len(items)} images, " +
          ", ".join(f"{CLASSES[c]} {sum((l == c).sum() for l in labs):,}" for c in range(6)
                    if sum((l == c).sum() for l in labs)))

## Stage 4 — train from the round-1 checkpoint

CE (ignore 255) + soft Dice, AdamW, one-cycle. The **last** step is reported.

In [ ]:
def dice_loss(logits, y):
    ok = (y != IGNORE)
    p = logits.softmax(1) * ok[:, None]
    oh = F.one_hot(y.clamp(max=len(CLASSES) - 1), len(CLASSES)).permute(0, 3, 1, 2) * ok[:, None]
    present = oh.sum((0, 2, 3)) > 0
    inter = (p * oh).sum((0, 2, 3)); den = p.sum((0, 2, 3)) + oh.sum((0, 2, 3))
    return (1 - (2 * inter + 1) / (den + 1))[present].mean()


def fmt(ev):
    return "  ".join(f"{n} {ev[n]['miou']:.3f}" for n in VAL)


model = smp.Unet("resnet34", encoder_weights=None, classes=len(CLASSES))
model.load_state_dict(torch.load(HERE / init_ckpt, map_location="cpu"))
model = model.to(DEV)

before, before_preds = evaluate(model)
history = [{"step": 0, "eval": before}]
print(f"step    0  (round-1 model)  {fmt(before)}")

opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.05) if steps else None
rng = np.random.default_rng(seed)
t0, losses = time.time(), {"sat": [], "drone": []}
for step in range(1, steps + 1):
    model.train()
    dom = "drone" if rng.random() < drone_p else "sat"
    x, y = (batch_of(drone_sample, rng, drone_batch) if dom == "drone"
            else batch_of(sat_sample, rng, sat_batch))
    x, y = x.to(DEV), y.to(DEV)
    out = model(x)
    loss = F.cross_entropy(out, y, ignore_index=IGNORE) + dice_loss(out, y)
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    losses[dom].append(loss.item())
    if step % eval_every == 0 or step == steps:
        ev, _ = evaluate(model)
        history.append({"step": step, "eval": ev,
                        "loss": {k: float(np.mean(v)) if v else None for k, v in losses.items()}})
        el = time.time() - t0
        print(f"step {step:4d}  loss sat {np.mean(losses['sat'] or [np.nan]):.3f} "
              f"drone {np.mean(losses['drone'] or [np.nan]):.3f}  {fmt(ev)}  [{el/60:.0f} min]")
        losses = {"sat": [], "drone": []}

train_min = (time.time() - t0) / 60
if steps:                      # steps=0 is an evaluation-only run of init_ckpt
    torch.save(model.state_dict(), HERE / f"model-{tag}.pt")

## Stage 5 — numbers, before vs after

In [ ]:
after, after_preds = evaluate(model)
for name in VAL:
    b, a = before[name], after[name]
    print(f"\n{name}")
    print(f"  {'class':>10s} {'round 1':>8s} {'round 2':>8s}")
    for c in b["iou"]:
        print(f"  {c:>10s} {b['iou'][c]:>8.3f} {a['iou'][c]:>8.3f}")
    print(f"  {'mIoU':>10s} {b['miou']:>8.3f} {a['miou']:>8.3f}")
    if "agreement" in a and a["agreement"] is not None:
        print(f"  {'agreement':>10s} {b['agreement']:>8.3f} {a['agreement']:>8.3f}")

results = {"tag": tag, "init": init_ckpt,
           "params": {"drone_p": drone_p, "drone_gsd": drone_gsd, "steps": steps, "lr": lr,
                      "sat_crop": sat_crop, "sat_batch": sat_batch, "drone_crop": drone_crop,
                      "drone_batch": drone_batch, "rare_crop_p": rare_crop_p, "seed": seed, "device": str(DEV)},
           "splits": {"sat_train": [(s["loc"], s["date"]) for s in sat_train],
                      "sat_holdout_village": sat_holdout_village, "birky_val_rows": BIRKY_VAL,
                      "drone_train": [d["name"] for d in drone_train], "drone_val": [d["name"] for d in drone_val],
                      "drone_dropped": [d["name"] for d in drone if d["split"] == "dropped"]},
           "train_minutes": train_min, "before": before, "after": after, "history": history}
(HERE / f"results-{tag}.json").write_text(json.dumps(results, indent=1, default=float))

## Look at it

In [ ]:
steps_ = [h["step"] for h in history]
fig, axes = plt.subplots(1, len(VAL), figsize=(6 * len(VAL), 4))
for a, name in zip(axes, VAL):
    for c in history[0]["eval"][name]["iou"]:
        a.plot(steps_, [h["eval"][name]["iou"][c] for h in history], ".-",
               color=PAL[CLASSES.index(c)] / 255, label=c)
    a.plot(steps_, [h["eval"][name]["miou"] for h in history], "k--", lw=2, label="mIoU")
    a.set_title(name); a.set_ylim(0, 1); a.grid(alpha=.3); a.set_xlabel("step (0 = round-1 model)")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIGS / f"{tag}-curve.png", dpi=110); plt.show()

In [ ]:
# Drone hold-out: frame / label / round-1 prediction / round-2 prediction
show = np.linspace(0, len(drone_val) - 1, min(8, len(drone_val))).astype(int)
fig, axes = plt.subplots(len(show), 4, figsize=(16, 2.4 * len(show) + 1))
for r, i in enumerate(show):
    im, lb, _ = VAL["drone-val"][i]
    panels = [(im, "frame"), (colour(lb), "label"),
              (colour(before_preds["drone-val"][i]), "round 1"), (colour(after_preds["drone-val"][i]), "round 2")]
    for a, (p, t) in zip(axes[r], panels):
        a.imshow(p); a.set_xticks([]); a.set_yticks([])
        if r == 0: a.set_title(t)
    axes[r, 0].set_ylabel(f"{drone_val[i]['name'][-8:]}\n{drone_val[i]['alt']:.0f} m", fontsize=8)
handles = [plt.Rectangle((0, 0), 1, 1, color=PAL[c] / 255) for c in range(6)]
fig.legend(handles, CLASSES, loc="lower center", ncol=6)
plt.tight_layout(rect=(0, .03, 1, 1)); plt.savefig(FIGS / f"{tag}-drone-val.png", dpi=90, bbox_inches="tight"); plt.show()

In [ ]:
# Unseen village: each date, image / label / round-1 / round-2 (downsampled x3 for display)
items = VAL["bushcha"]
fig, axes = plt.subplots(len(items), 4, figsize=(22, 4.4 * len(items)))
axes = np.atleast_2d(axes)
for r, (im, lb, _) in enumerate(items):
    panels = [(im[::3, ::3], "image"), (colour(lb[::3, ::3]), "label"),
              (colour(before_preds["bushcha"][r][::3, ::3]), "round 1"),
              (colour(after_preds["bushcha"][r][::3, ::3]), "round 2")]
    for a, (p, t) in zip(axes[r], panels):
        a.imshow(p); a.axis("off"); a.set_title(f"{t}", fontsize=10)
plt.tight_layout(); plt.savefig(FIGS / f"{tag}-bushcha.png", dpi=80, bbox_inches="tight"); plt.show()

## Stage 6 — BatchNorm check

Each training step is all-satellite or all-drone, and BatchNorm's running mean
and variance are an exponential average over recent batches. At eval time the
model normalises with whatever mix of domains came last — so satellite scores
can swing with the order of the last few batches, not with what was learned.

Test, no retraining: reset every BN layer's running statistics, re-estimate them
(cumulative average, no gradients) from 60 batches of satellite only, drone only,
or both alternating, and evaluate each variant on all three hold-outs. If
satellite scores recover under satellite or mixed statistics, the swings were BN,
and the trained weights are better than the logged curve says.

In [ ]:
import copy


def recalibrate(model, which, n=60):
    m = copy.deepcopy(model)
    for mod in m.modules():
        if isinstance(mod, torch.nn.modules.batchnorm._BatchNorm):
            mod.reset_running_stats(); mod.momentum = None       # cumulative average over the n batches
    m.train()
    r = np.random.default_rng(123)
    with torch.no_grad():
        for i in range(n):
            dom = which if which != "mixed" else ("sat" if i % 2 == 0 else "drone")
            x, _ = (batch_of(sat_sample, r, sat_batch) if dom == "sat"
                    else batch_of(drone_sample, r, drone_batch))
            m(x.to(DEV))
    return m.eval()


variants = {"as trained": model, "BN: satellite": recalibrate(model, "sat"),
            "BN: drone": recalibrate(model, "drone"), "BN: mixed": recalibrate(model, "mixed")}
bn_check = {}
for vname, m in variants.items():
    bn_check[vname], _ = evaluate(m)

print(f"{'BN statistics':>14s} " + " ".join(f"{n:>12s}" for n in VAL))
for vname, ev in bn_check.items():
    print(f"{vname:>14s} " + " ".join(f"{ev[n]['miou']:>12.3f}" for n in VAL))
for n in VAL:
    print(f"\n{n}: " + " | ".join(f"{v}: " + " ".join(f"{c[:4]} {x:.2f}" for c, x in bn_check[v][n]["iou"].items())
                              for v in bn_check))
(HERE / f"results-{tag}-bncheck.json").write_text(json.dumps(bn_check, indent=1, default=float))

## Verdict

Written into `notes.md` after looking at the figures.